# 🧼 Checkpoint 8 — Focus Preprocessing (Nettoyage & Mise à l'Échelle)

## Module Data Science — Preprocessing

> **Prérequis :** cours *Nettoyage des Données & Mise à l'Échelle* (`05-...md`).

Notebook **corrigé** du checkpoint 8. Trois parties :
- **A.** Nettoyage de données réelles (`fiche_employes.csv`)
- **B.** Feature Scaling — comparaison des 4 échelles (`demandes_credit.csv`)
- **C.** Pipeline de preprocessing anti-fuite + modèle

> 💡 Les jeux de données sont ceux du repo. Le résolveur de chemin ci-dessous trouve les CSV
> quel que soit le dossier d'exécution (repo local ou upload Colab).

In [ ]:
!pip install scikit-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
np.random.seed(42)

def trouver(nom):
    """Cherche un CSV dans les emplacements probables du repo (ou dossier courant)."""
    candidats = [nom,
                 f'../../03-data-science/TP/{nom}',
                 f'../TP/{nom}',
                 f'04-machine-learning/TP/{nom}',
                 f'03-data-science/TP/{nom}']
    for c in candidats:
        if Path(c).exists():
            return c
    raise FileNotFoundError(f"{nom} introuvable — uploadez-le à côté du notebook.")

print('Environnement prêt ✅')

---
# PARTIE A — Nettoyer des données réelles

`fiche_employes.csv` contient des défauts typiques du terrain : noms de colonnes incohérents
(typo `Fist Name`), adresses multilignes, téléphones variés, intitulés de poste en tuple-texte
`"('Chef de projet finance',)"`, villes en casses différentes, valeurs manquantes.

## A.1 — Charger & diagnostiquer la qualité

In [ ]:
df = pd.read_csv(trouver('fiche_employes.csv'))
print('Dimensions :', df.shape)
print('\nColonnes brutes :', list(df.columns))
print('\nValeurs manquantes par colonne :')
print(df.isna().sum())
print('\nDoublons exacts :', df.duplicated().sum())
df.head(3)

In [ ]:
df.info()

## A.2 — Nettoyage complet

On travaille sur une **copie** (données brutes préservées), dans l'ordre méthodique :
colonnes → types → texte → inexactitudes → manquants → doublons → outliers.

In [ ]:
df_clean = df.copy()   # ✅ ne jamais toucher à l'original

# 1. Noms de colonnes : minuscules, sans espaces, corriger les typos
df_clean.columns = df_clean.columns.str.strip().str.lower().str.replace(' ', '_')
df_clean = df_clean.rename(columns={'fist_name': 'first_name',
                                    'djob_tilte': 'job_title'})

# 2. Types corrects
df_clean['salary'] = pd.to_numeric(df_clean['salary'], errors='coerce')
df_clean['join_date'] = pd.to_datetime(df_clean['join_date'], errors='coerce')

# 3. Texte : ville normalisée + poste extrait du tuple-texte
df_clean['ville'] = df_clean['ville'].str.strip().str.lower()
df_clean['job_title'] = df_clean['job_title'].str.extract(r"'([^']+)'")

# 4. Inexactitudes : âge hors bornes plausibles -> manquant
df_clean.loc[(df_clean['age'] > 100) | (df_clean['age'] < 16), 'age'] = np.nan

# 5. Manquants : médiane (numérique), mode (catégoriel)
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median()).astype(int)
df_clean['salary'] = df_clean['salary'].fillna(df_clean['salary'].median())
for col in ['department', 'ville', 'gender', 'employment_status']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# 6. Dédupliquer sur l'email (APRÈS standardisation)
df_clean['email'] = df_clean['email'].str.strip().str.lower()
avant = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['email'], keep='first')
print(f'Doublons email supprimés : {avant - len(df_clean)}')

## A.3 — Détecter et traiter les outliers de `salary` (méthode IQR)

Un salaire élevé est **plausible** (cadre dirigeant) → on **cappe** plutôt que de supprimer.

In [ ]:
# Snapshot du salaire AVANT capping — fait UNE seule fois, donc la cellule
# reste correcte même si on la ré-exécute (sinon 'avant' = déjà cappé).
if 'salary_brut' not in df_clean.columns:
    df_clean['salary_brut'] = df_clean['salary']

# Bornes IQR calculées sur le BRUT
Q1, Q3 = df_clean['salary_brut'].quantile([0.25, 0.75])
IQR = Q3 - Q1
bas, haut = Q1 - 1.5*IQR, Q3 + 1.5*IQR
n_out = int(((df_clean['salary_brut'] < bas) | (df_clean['salary_brut'] > haut)).sum())
print(f'Bornes IQR : [{bas:,.0f} ; {haut:,.0f}]  —  outliers détectés : {n_out}')
print(f"Salaire max : {df_clean['salary_brut'].max():,.0f} (brut)  ->  {haut:,.0f} (après capping)")

# Capping à partir du BRUT (idempotent)
df_clean['salary'] = df_clean['salary_brut'].clip(bas, haut)

# Comparaison AVANT (brut) vs APRÈS (cappé) — mêmes bornes d'axe pour bien voir l'effet
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharex=True)
df_clean['salary_brut'].plot.box(ax=axes[0], vert=False); axes[0].set_title('Salary AVANT capping')
df_clean['salary'].plot.box(ax=axes[1], vert=False); axes[1].set_title('Salary APRÈS capping')
plt.tight_layout(); plt.show()

## A.4 — Vérification finale du nettoyage

In [ ]:
print('Dimensions finales :', df_clean.shape)
print('\nManquants restants (colonnes clés) :')
print(df_clean[['age','salary','department','ville','gender','employment_status']].isna().sum())
print('\nExemples de postes nettoyés :', df_clean['job_title'].dropna().unique()[:4])
df_clean.head(3)

---
# PARTIE B — Feature Scaling : comparer les 4 échelles

Sur `demandes_credit.csv`, on compare **MaxAbs**, **MinMax**, **Standard** et **Robust**
sur des variables aux amplitudes très différentes (`age` vs `revenu_mensuel`).

In [ ]:
credit = pd.read_csv(trouver('demandes_credit.csv'))
cols = ['age', 'revenu_mensuel', 'montant_demande']
X = credit[cols].dropna()

print('AVANT tout scaling — noter les écarts d\'amplitude :')
X.describe().loc[['min', 'max', 'mean', 'std']].round(1)

In [ ]:
from sklearn.preprocessing import (MaxAbsScaler, MinMaxScaler,
                                   StandardScaler, RobustScaler)

scalers = {'MaxAbs': MaxAbsScaler(), 'MinMax': MinMaxScaler(),
           'Standard': StandardScaler(), 'Robust': RobustScaler()}

for nom, sc in scalers.items():
    Xs = pd.DataFrame(sc.fit_transform(X), columns=cols)
    stats = Xs.describe().loc[['min', 'max', 'mean', 'std']].round(3)
    print(f'\n=== {nom} ===')
    print(stats)

print('\n👉 MinMax -> [0,1] exact | Standard -> moyenne≈0, std≈1 | Robust -> médiane centrée (résiste aux outliers)')

In [ ]:
# Visualiser l'effet sur 'revenu_mensuel'
fig, axes = plt.subplots(1, 5, figsize=(16, 3), sharey=False)
X['revenu_mensuel'].plot.hist(ax=axes[0], bins=30); axes[0].set_title('Brut')
for ax, (nom, sc) in zip(axes[1:], scalers.items()):
    vals = sc.fit_transform(X)[:, 1]   # colonne revenu_mensuel
    ax.hist(vals, bins=30); ax.set_title(nom)
plt.tight_layout(); plt.show()

print('La FORME est préservée ; seule l\'ÉCHELLE change.')

### 🔬 Faut-il scaler ici ?

| Algorithme | Scaling ? | Pourquoi |
|------------|-----------|----------|
| **Random Forest** | ❌ Non | Découpe par seuils, insensible à l'échelle |
| **KNN / SVM / régression** | ✅ Oui | Basés sur distances / gradient → le revenu écraserait l'âge sans scaling |

---
# PARTIE C — Pipeline de preprocessing complet (anti-fuite)

On assemble **imputation + scaling + encodage + modèle** dans un `Pipeline` avec `ColumnTransformer`.
Tout est `fit` **sur le train uniquement** → aucune fuite de données possible.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

X = credit.drop(columns=['credit_accorde'])
y = credit['credit_accorde']

# 1. Split stratifié AVANT toute transformation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('Train :', X_train.shape, '| Test :', X_test.shape)

In [ ]:
# 2. Colonnes par type (détection automatique)
num = X.select_dtypes('number').columns.tolist()
cat = X.select_dtypes('object').columns.tolist()
print('Numériques :', num)
print('Catégorielles :', cat)

pipe_num = Pipeline([('imputer', SimpleImputer(strategy='median')),
                     ('scaler',  StandardScaler())])
pipe_cat = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                     ('onehot',  OneHotEncoder(handle_unknown='ignore'))])

prep = ColumnTransformer([('num', pipe_num, num),
                          ('cat', pipe_cat, cat)])

# 3. Pipeline complet : preprocessing + modèle
model = Pipeline([('prep', prep),
                  ('clf',  LogisticRegression(max_iter=1000))])

In [ ]:
# 4. Entraîner sur le TRAIN, évaluer sur le TEST
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f'Accuracy : {accuracy_score(y_test, y_pred):.3f}')
print(f'F1       : {f1_score(y_test, y_pred):.3f}')
print()
print(classification_report(y_test, y_pred))

In [ ]:
# 5. Preuve d'anti-fuite : le scaler n'a appris QUE sur le train
scaler_ajuste = model.named_steps['prep'].named_transformers_['num'].named_steps['scaler']
moy_pipeline = scaler_ajuste.mean_[num.index('revenu_mensuel')]
moy_train    = X_train['revenu_mensuel'].mean()
moy_all      = X['revenu_mensuel'].mean()
print(f"Moyenne apprise par le pipeline : {moy_pipeline:,.1f}")
print(f"Moyenne du TRAIN seul           : {moy_train:,.1f}   <- doit correspondre")
print(f"Moyenne de TOUT le dataset      : {moy_all:,.1f}   <- ne correspond PAS (sinon = fuite)")

In [ ]:
# 6. Bonus : validation croisée sur le train (5 folds)
cv = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
print(f'F1 en validation croisée : {cv.mean():.3f} ± {cv.std():.3f}')
print('Écart-type faible = modèle stable ✅')

---
# ✅ Synthèse du checkpoint

| Partie | Compétence validée |
|--------|--------------------|
| **A** | Diagnostiquer + nettoyer des données réelles (types, texte, manquants, doublons, outliers) |
| **B** | Choisir et comparer les mises à l'échelle ; savoir quand scaler |
| **C** | Pipeline + ColumnTransformer anti-fuite, prêt pour la production |

> 🔑 **La règle d'or** : `fit` sur le **train uniquement**. Le Pipeline garantit l'anti-fuite par construction.

### 🚀 Pour aller plus loin
- Remplacez `LogisticRegression` par un `RandomForestClassifier` (retirez le scaler : inutile pour un arbre).
- Ajoutez un `GridSearchCV` sur le pipeline pour régler les hyperparamètres (cours 03).
- Testez `KNNImputer` au lieu de `SimpleImputer` pour l'imputation numérique.

---
*📘 Module Data Science — Checkpoint 8 : Focus Preprocessing | Bootcamp Data Science*